# Notebook 2: Metadata Extraction (EXIF / TIFF / IPTC)

This notebook extracts structured metadata from TIFF image files, conforming to **IPTC** and **Dublin Core** standards.

### Metadata extracted per image:
| Category | Fields |
|----------|--------|
| **File-level** | Filename, file size, format, MIME type |
| **EXIF / TIFF** | Dimensions, resolution, compression, photometric interpretation, software, dates |
| **IPTC** (if available) | Object name, caption/abstract, keywords |

### Why standardize?
- **Interoperability** — metadata works across systems and platforms
- **Discoverability** — supports search, retrieval, and classification
- **Archival quality** — long-term preservation and digital asset management compliance

## Configuration

In [ ]:
from pathlib import Path

# ============================================================
# CONFIGURATION — Edit these paths before running
# ============================================================

# Folder containing TIFF images to analyze
IMAGE_FOLDER = Path(r"path/to/your/tiff/images")  # e.g. Path(r"C:\Downloads\Numbered Scans_1")

# Output CSV path
OUTPUT_CSV = Path(r"path/to/output/metadata.csv")  # e.g. Path(r"C:\Downloads\metadata_output.csv")

# Maximum number of images to process (set to None for all)
MAX_IMAGES = 1000

print(f"Image folder: {IMAGE_FOLDER}")
print(f"Output CSV:   {OUTPUT_CSV}")

## Imports

In [ ]:
import os
import tifffile as tiff
from PIL import Image, ExifTags
import pandas as pd

try:
    from iptcinfo3 import IPTCInfo
    HAS_IPTC = True
    print("iptcinfo3 available — IPTC metadata will be extracted.")
except Exception:
    HAS_IPTC = False
    print("iptcinfo3 not installed — IPTC metadata will be skipped. Install with: pip install iptcinfo3")

## Extraction Functions

In [ ]:
def human_size(num_bytes: int) -> str:
    """Convert a byte count to a human-readable string."""
    units = ["B", "KiB", "MiB", "GiB"]
    size = float(num_bytes)
    for u in units:
        if size < 1024.0:
            return f"{size:.1f} {u}"
        size /= 1024.0
    return f"{size:.1f} TiB"


def probe_file(path: Path) -> dict:
    """Extract file-level attributes."""
    info = {}
    st = path.stat()
    info["Filename"] = path.name
    info["File Size"] = human_size(st.st_size)
    info["File Type"] = path.suffix.upper().lstrip(".")
    info["File Type Extension"] = path.suffix.lstrip(".")
    info["MIME Type"] = "image/tiff" if path.suffix.lower().startswith(".tif") else None
    return info


def extract_tiff_tags(path: Path) -> dict:
    """Extract TIFF-specific tags using tifffile."""
    data = {}
    try:
        with tiff.TiffFile(str(path)) as tf:
            page = tf.pages[0]
            data["Image Width"] = page.imagewidth
            data["Image Height"] = page.imagelength
            data["Bits Per Sample"] = page.bitspersample
            data["Compression"] = page.compression.name if hasattr(page.compression, "name") else page.compression
            data["Samples Per Pixel"] = page.samplesperpixel
            data["Photometric Interpretation"] = page.photometric.name if hasattr(page.photometric, "name") else page.photometric
            data["Rows Per Strip"] = page.rowsperstrip
            data["X Resolution"] = page.tags.get("XResolution").value if "XResolution" in page.tags else None
            data["Y Resolution"] = page.tags.get("YResolution").value if "YResolution" in page.tags else None
            data["Resolution Unit"] = page.tags.get("ResolutionUnit").value if "ResolutionUnit" in page.tags else None
    except Exception:
        pass
    return data


def extract_exif(path: Path) -> dict:
    """Extract EXIF metadata via Pillow."""
    data = {}
    try:
        with Image.open(path) as im:
            raw = im.getexif() or {}
            tag_map = {ExifTags.TAGS.get(k, str(k)): v for k, v in dict(raw).items()}
            for k in ["Software", "ModifyDate", "Artist", "Copyright"]:
                if k in tag_map:
                    data[k] = tag_map[k]
            data["Exif Image Width"] = im.width
            data["Exif Image Height"] = im.height
    except Exception:
        pass
    return data


def extract_iptc(path: Path) -> dict:
    """Extract IPTC metadata if iptcinfo3 is available."""
    if not HAS_IPTC:
        return {}
    try:
        info = IPTCInfo(str(path), force=True)
        out = {}
        if info.get("object name"):
            out["Object Name"] = info.get("object name")
        if info.get("caption/abstract"):
            out["Caption"] = info.get("caption/abstract")
        if info.get("keywords"):
            out["Keywords"] = info.get("keywords")
        return out
    except Exception:
        return {}


def analyze(path: Path) -> dict:
    """Combine all metadata sections into a flat dict with section-prefixed keys."""
    sections = {
        "File": probe_file(path),
        "EXIF": {**extract_tiff_tags(path), **extract_exif(path)},
    }
    if HAS_IPTC:
        iptc = extract_iptc(path)
        if iptc:
            sections["IPTC"] = iptc

    flat = {}
    for sec, vals in sections.items():
        for k, v in vals.items():
            flat[f"{sec}.{k}"] = v
    return flat

## Run Extraction

In [ ]:
files = sorted(IMAGE_FOLDER.glob("*.tif"))
if MAX_IMAGES:
    files = files[:MAX_IMAGES]

print(f"Found {len(files)} TIFF files to process...\n")

records = []
for i, f in enumerate(files, start=1):
    if i % 100 == 0 or i == 1:
        print(f"[{i}/{len(files)}] Processing {f.name}...")
    record = analyze(f)
    records.append(record)

df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False)

print(f"\nDone. {len(records)} records saved to: {OUTPUT_CSV}")
df.head()